In [1]:
import xarray as xr
from netCDF4 import Dataset
import cfgrib
import eccodes
import pandas as pd
import numpy as np
import pygrib

import os
from pathlib import Path

import logging
from os import scandir
import re
import subprocess
import time
import zipfile
import xarray as xr
import cfgrib
from netCDF4 import Dataset
from netCDF4 import num2date, date2num

In [2]:
folder = '/fs/yedoma/data/globsim/n60/test_split/'

In [3]:
def split_resl_nc(f, sa, sf, time_var):
    cmd1 = f"nccopy -V {time_var},latitude,longitude,ssrd,strd,tp {f} {sf}"
    cmd2 = f"nccopy -V {time_var},latitude,longitude,d2m,t2m,tco3,tcwv,u10,v10 {f} {sa}"
    # logger.debug(cmd1)
    p1 = subprocess.Popen(cmd1.split(" "))
    p1.wait()
    # logger.debug(cmd2)
    subprocess.Popen(cmd2.split(" "))
    # if Path(sf).exists() and Path(sa).exists():
        # logger.debug(f"Removing {f}")
        # Path(f).unlink()
    cmd1 = f'ncks -C -O -x -v time {sf} {sf}'
    cmd2 = f'ncks -C -O -x -v time {sa} {sa}'
    # logger.debug(cmd1)
    p1 = subprocess.Popen(cmd1.split(" "))
    p1.wait()
    # logger.debug(cmd2)
    p2 = subprocess.Popen(cmd2.split(" "))
    p2.wait()

def convert_time_sa(f, overwrite=True, time_var='valid_time'):
    sa_pattern = re.compile(r"era5_sa_(\d{8}_to_\d{8}).nc")
    orig = sa_pattern.sub(r"era5_sa_\1.nc", f)
    # logger.debug(f"Converting time units for {Path(f).name}")
    print(orig)

    # if not overwrite and Path(orig).exists():
    #     print(f"Skipping {orig}")
    #     return

    nc = Dataset(orig, mode='r+')

    newunit = 'seconds since 1970-01-01'
    timevar = nc.variables[time_var]
    timein = timevar[:]
    datesin = num2date(timein,timevar.units)
    timevar.setncattr('units',newunit)
    timevar[:] = date2num(datesin,newunit)
    nc.close()

def convert_time_sf(f, overwrite=True, time_var='valid_time'):
    sa_pattern = re.compile(r"era5_sf_(\d{8}_to_\d{8}).nc")
    orig = sa_pattern.sub(r"era5_sf_\1.nc", f)
    # logger.debug(f"Converting time units for {Path(f).name}")
    print(orig)

    # if not overwrite and Path(orig).exists():
    #     print(f"Skipping {orig}")
    #     return

    nc = Dataset(orig, mode='r+')

    newunit = 'seconds since 1970-01-01'
    timevar = nc.variables[time_var]
    timein = timevar[:]
    datesin = num2date(timein,timevar.units)
    timevar.setncattr('units',newunit)
    timevar[:] = date2num(datesin,newunit)
    nc.close()

def convert_grib2nc_to(f, overwrite=False):
    pl_pattern = re.compile(r"era5_to.grib")
    print(pl_pattern)
    new_file = pl_pattern.sub(r"era5_to.nc", f)
    print(new_file)
    print(f, overwrite, Path(new_file).exists())
    if not overwrite and Path(new_file).exists():
        print(f"Skipping {new_file}")
        return
    else:
        print('a')

def rename_to_dir(dir):
    pl_pattern = re.compile(r"era5_to.grib")
    files = [str(f) for f in Path(dir).iterdir() if pl_pattern.search(str(f))]
    for f in files:
        convert_grib2nc_to(f)

In [17]:
dir = '/fs/yedoma/data/globsim/n60/era5/'
orig = re.compile(r"era5_re_resl_(\d{8}_to_\d{8}).grib")
files = [str(f) for f in Path(dir).iterdir()]
matched_files = [f for f in files if orig.search(f)]
matched_files_re_resl = [ff for ff in [f.replace('.grib', '.nc') for f in matched_files] if Path(ff).exists()]
matched_files_sa = [ff for ff in [f.replace('re_resl', 'sa') for f in matched_files_re_resl] if Path(ff).exists()]
matched_files_sf = [ff for ff in [f.replace('re_resl', 'sf') for f in matched_files_re_resl] if Path(ff).exists()]

In [18]:
matched_files_sa

['/fs/yedoma/data/globsim/n60/era5/era5_sa_20250701_to_20250731.nc']

In [13]:
[ff for ff in [f.replace('.grib', '.nc') for f in matched_files] if Path(ff).exists()]

['/fs/yedoma/data/globsim/n60/era5/era5_re_resl_20250701_to_20250731.nc']

In [17]:
rename_to_dir(folder)

re.compile('era5_to.grib')
/fs/yedoma/data/globsim/n60/test_split/era5_to.nc
/fs/yedoma/data/globsim/n60/test_split/era5_to.grib False True
Skipping /fs/yedoma/data/globsim/n60/test_split/era5_to.nc


In [4]:
files = [199109, 202501, 202502, 202503, 202504, 202505, 202506, 202507]

In [6]:
for f in files:
    for i in ['sa', 'sf', 'pl']:
        path_full = f'/fs/yedoma/data/globsim/n60/era5/era5_{i}_{f}01_to_{f}31.nc'
        if os.path.exists(path_full):
            nc = Dataset(path_full)
            if not nc['valid_time'].units == 'seconds since 1970-01-01':
                print(f, i)

199109 sf
202501 sa
202501 sf
202502 sa
202502 sf
202503 sa
202503 sf
202504 sa
202504 sf
202505 sa
202505 sf
202506 sa
202506 sf
202507 sa
202507 sf


In [8]:
for f in files:
    path_re_resl = folder+f'era5_re_resl_{f}01_to_{f}31.nc'
    path_sa = folder+f'era5_sa_{f}01_to_{f}31.nc'
    path_sf = folder+f'era5_sf_{f}01_to_{f}31.nc'
    split_resl_nc(path_re_resl, path_sa, path_sf, 'valid_time')
    convert_time_sa(path_sa)
    convert_time_sf(path_sf)

/fs/yedoma/data/globsim/n60/test_split/era5_sa_19910901_to_19910931.nc
/fs/yedoma/data/globsim/n60/test_split/era5_sf_19910901_to_19910931.nc
/fs/yedoma/data/globsim/n60/test_split/era5_sa_20250101_to_20250131.nc
/fs/yedoma/data/globsim/n60/test_split/era5_sf_20250101_to_20250131.nc
/fs/yedoma/data/globsim/n60/test_split/era5_sa_20250201_to_20250231.nc
/fs/yedoma/data/globsim/n60/test_split/era5_sf_20250201_to_20250231.nc
/fs/yedoma/data/globsim/n60/test_split/era5_sa_20250301_to_20250331.nc
/fs/yedoma/data/globsim/n60/test_split/era5_sf_20250301_to_20250331.nc
/fs/yedoma/data/globsim/n60/test_split/era5_sa_20250401_to_20250431.nc
/fs/yedoma/data/globsim/n60/test_split/era5_sf_20250401_to_20250431.nc
/fs/yedoma/data/globsim/n60/test_split/era5_sa_20250501_to_20250531.nc
/fs/yedoma/data/globsim/n60/test_split/era5_sf_20250501_to_20250531.nc
/fs/yedoma/data/globsim/n60/test_split/era5_sa_20250601_to_20250631.nc
/fs/yedoma/data/globsim/n60/test_split/era5_sf_20250601_to_20250631.nc
/fs/ye

In [9]:
for f in files:
    for i in ['sa', 'sf']:
        path_full = f'/fs/yedoma/data/globsim/n60/test_split/era5_{i}_{f}01_to_{f}31.nc'
        if os.path.exists(path_full):
            nc = Dataset(path_full)
            if not nc['valid_time'].units == 'seconds since 1970-01-01':
                print(f, i)

In [10]:
for f in files:
    for i in ['sa', 'sf', 'pl']:
        path_full = f'/fs/yedoma/data/globsim/n60/era5/era5_{i}_{f}01_to_{f}31.nc'
        if os.path.exists(path_full):
            nc = Dataset(path_full)
            if not nc['valid_time'].units == 'seconds since 1970-01-01':
                print(f, i)

In [7]:
path_re_resl = folder+'era5_re_resl_20121101_to_20121131.nc'
path_sa = folder+'era5_sa_20121101_to_20121131.nc'
path_sf = folder+'era5_sf_20121101_to_20121131.nc'

In [8]:
split_resl_nc(path_re_resl, path_sa, path_sf, 'valid_time')

In [9]:
convert_time_sa(path_sa)

/fs/yedoma/data/globsim/n60/test_split/era5_sa_20121101_to_20121131.nc


In [10]:
convert_time_sf(path_sf)

/fs/yedoma/data/globsim/n60/test_split/era5_sf_20121101_to_20121131.nc


In [9]:
path_n60 = '/fs/yedoma/data/globsim/Salluit/era5/'

In [10]:
for year in range(2025,2026,1):
    for month in range(1,13,1):
        for i in ['sa', 'sf', 'pl']:
            path_full = f"{path_n60}era5_{i}_{year}{'0' if month<10 else ''}{month}01_to_{year}{'0' if month<10 else ''}{month}31.nc"
            if os.path.exists(path_full):
                print(path_full)
                nc = Dataset(path_full)
                if not nc['valid_time'].units == 'seconds since 1970-01-01':
                    print(year, month, i)
    print(f'{year} is done.')

/fs/yedoma/data/globsim/Salluit/era5/era5_sa_20250101_to_20250131.nc
/fs/yedoma/data/globsim/Salluit/era5/era5_sf_20250101_to_20250131.nc
/fs/yedoma/data/globsim/Salluit/era5/era5_pl_20250101_to_20250131.nc
/fs/yedoma/data/globsim/Salluit/era5/era5_sa_20250201_to_20250231.nc
/fs/yedoma/data/globsim/Salluit/era5/era5_sf_20250201_to_20250231.nc
/fs/yedoma/data/globsim/Salluit/era5/era5_pl_20250201_to_20250231.nc
/fs/yedoma/data/globsim/Salluit/era5/era5_sa_20250301_to_20250331.nc
/fs/yedoma/data/globsim/Salluit/era5/era5_sf_20250301_to_20250331.nc
/fs/yedoma/data/globsim/Salluit/era5/era5_pl_20250301_to_20250331.nc
/fs/yedoma/data/globsim/Salluit/era5/era5_sa_20250401_to_20250431.nc
/fs/yedoma/data/globsim/Salluit/era5/era5_sf_20250401_to_20250431.nc
/fs/yedoma/data/globsim/Salluit/era5/era5_pl_20250401_to_20250431.nc
2025 is done.


In [26]:
nc=Dataset(path_n60+'era5_sa_20121101_to_20121131.nc')
nc=Dataset(path_n60+'era5_sa_20250701_to_20250731.nc')

In [27]:
nc['valid_time'].units == 'seconds since 1970-01-01'

False